#### Importing Dependencies.
Make sure to create an env using <b> python 3.10 </b> to prevent facing any library issue.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import re
import time
import hazm
import csv

import pandas as pd
from tqdm import tqdm

__________________________________________________
#### Loading Datasets Using Huggingface Datasets

In [ ]:
# === ------ Define directory path (cache_dir) to dataset ------- === #
# === This code will save and load dataset from your defined path === #
data_path = os.path.join("C:/Users/Amir/Desktop/Datasets")

import warnings
warnings.filterwarnings("ignore")
from datasets import load_dataset

try:
    farsi_corpus = load_dataset("ali619/corpus-dataset-normalized-for-persian-farsi", cache_dir=data_path)
    farsi_news = load_dataset("community-datasets/farsi_news", cache_dir=data_path)
    farsi_wiki_qa = load_dataset("fibonacciai/Persian-Wikipedia-QA", cache_dir=data_path)
    farsi_blogs = load_dataset("MaralGPT/persian_blogs", cache_dir=data_path)
    farsi_quotes = load_dataset("MaralGPT/persian_quotes", cache_dir=data_path)
    farsi_wiki = load_dataset("MaralGPT/persian-wikipedia", cache_dir=data_path)
    farsi_qa = load_dataset("mshojaei77/persian_sft_qa", cache_dir=data_path)
    farsi_stories = load_dataset("taesiri/tiny_stories-farsi", cache_dir=data_path)
except Exception as e:
    print(f"An error has been occured: {e}")

print("All datasets has been loaded successfully.")

________________________
#### Data Preprocessing
* Clean datasets like removing extra spaces and concatinating half-spaces words
* Converting persian texts into embeddings

In [ ]:
print("==========================================================================================")
print(f"* farsi 'corpus' - items and data shape: {farsi_corpus.shape}\n")

print(f"* farsi 'news' - items and data shape: {farsi_news.shape}\n")

print(f"* farsi 'wikipedia (QA)' - items and data shape: {farsi_wiki_qa.shape}\n")

print(f"* farsi 'blogs' - items and data shape: {farsi_blogs.shape}\n")

print(f"* farsi 'quotes' - items and data shape: {farsi_quotes.shape}\n")

print(f"* farsi 'wikipedia' - items and data shape: {farsi_wiki.shape}\n")

print(f"* farsi 'question answers' - items and data shape: {farsi_qa.shape}\n")

print(f"* farsi 'stories' - items and data shape: {farsi_stories.shape}")
print("==========================================================================================")

In [ ]:
# === Using 'hazm' library for text normalization === #
normalizer = hazm.normalizer.Normalizer()
# =================================================== #

def text_normalizer(text, junk_len=120, output='sentence'):
    # ============================================== #
    # === hazm normalizer works better, trust me === #
    # ============================================== #
    
    """text: your input text for normalization
       junk_len: minimum length of each sentence to delete references and meaningless sentences
       output: could be 'sentence' to have only 1 dimension string output
               or
               could be 'paragraph' to have all seperate sentences
    """
    
    # --- If 'wiki' is false, means that the text is not extracted from wikipedia --- #
    # --- Extracted texts from wikipedia have some especial format and are more complex for regex --- #
    
    normalized_text = normalizer.normalize(text)
    paragraphs = re.split("\n|\n\n", normalized_text)

    for i in range(len(paragraphs)):
        if len(paragraphs[i]) <= junk_len:
            paragraphs[i] = ""
        paragraphs[i] = re.sub(r"\s+([،:؛؟!,/])", r"\1", paragraphs[i])
        paragraphs[i] = re.sub(r"\s+", " ", paragraphs[i])
        paragraphs[i] = re.sub(r"\s+([*-_+])", "", paragraphs[i])
        paragraphs[i] = paragraphs[i].strip()

    paragraphs = list(filter(None, paragraphs))
    
    if output=='sentence':
        new_p = ''
        for p in paragraphs:
            new_p += ''.join(p)
        return(new_p)
    
    if output=='paragraph':
        return(paragraphs)


    
# === Saving the result of text normalization in a 'csv' file === #
save = False
file_name = "wiki_normalized"
if save:
    with open(f"{file_name}.csv", "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow([
            "url",         # URL
            "text",        # Text
            "reference",   # Reference
            "num_tokens"]) # Tokens

        for i in tqdm(range(len(farsi_wiki['train']['text']))):
            normalized_text = text_normalizer(farsi_wiki['train']['text'][i])

            writer.writerow([
                farsi_wiki['train']['url'][i], # URL
                normalized_text,               # Text
                "Wikipedia",                   # Reference
                # === Note: 1 Token is about 4 characters === #
                len(normalized_text) // 4])    # Tokens
    
    # === Post processing to remove NAN values === #
    df = pd.read_csv(f"{file_name}.csv")
    df = df.dropna(subset=["text"])
    df.to_csv(f"{file_name}.csv", index=False, encoding="utf-8-sig")
    print(f"Normalization is done and the results are saved in '{file_name}.csv'")

else:
    print(f"WARNING: save status is {save}.")

______________________
#### Chunking Dataset
* Convert processed texts into chunks
    * as far as we are using wikipedia as our dataset, we're gonna use parameters below
        * chunk size: 512
        * overlap: 64
        * top k: 5

In [ ]:
# === Visualization of the part of the normalized dataset === #
file_name = "wikipedia_corpus"
df = pd.read_csv(f"{file_name}.csv")
df.head(5)

__________________________
#### Creating Data Chunks

In [ ]:
CHUNK_SIZE = 512
TOP_K = 5
chunks = []
urls = []
single_chunk = []
count_tokens = 0

# Loop through all dataframes
for i in tqdm(range(len(df))):
    # Split each dataframe text into their sentences
    splitted_texts = re.split(r"(?<=[.!؟])\s+", df['text'][i])

    # Loop through number of sentences of dataframe number 'n'
    for sentence in splitted_texts:
        sentence_tokens = len(sentence) / 4

        # IF number of tokens from previous calculation + current sentence we are using
        # is less than the threshold, add sentences to a single chunk or list
        if count_tokens + sentence_tokens <= CHUNK_SIZE:
            single_chunk.append(sentence)
            count_tokens += sentence_tokens
        # ELSE, check the single chunk or list that we had
        # if it wasn't empty add it to our all chunks which has about 512 tokens
        else:
            if single_chunk:
                chunks.append(single_chunk)
            # use the last sentence of our last single chunk or list as overlap to bridge the gap
            overlap = single_chunk[-1] if single_chunk else ""
            
            # after we calculated overlap, add overlap (last sentence of the last chunk)
            # and the sentence we want to use right now to put it in a new chunk
            if overlap:
                # making first single chunk or list of our new chunk
                single_chunk = [overlap, sentence]
                # number of current tokens calculation
                count_tokens = len(overlap) / 4 + sentence_tokens
            else:
                # if we hadn't overlap, just reset the parameters for new chunk
                single_chunk = [sentence]
                count_tokens = sentence_tokens

# IF the loop has been ended but we didn't had the chance to add the calculated chunk
# to the list of our chunks, the code below would do that automatically
if single_chunk:
    chunks.append(single_chunk)
    
print(f"Chunks has been created.")
print(f"Number of chunks: '{len(chunks)}'")
print(f"Number of tokens: '{df['num_tokens'].sum()}'")

In [ ]:
file_name = "chunk_wiki_dataset"
with open(f"{file_name}.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["chunk"])

    for c in tqdm(chunks):
        writer.writerow(["".join(c)])
        
print(f"Dataset converted into chunks and saved as '{file_name}.csv'")